# Demonstração mínima do pipeline analítico (E1-E5)
**Caso de validação: detecção precoce de aliciamento online**

Este notebook foi preparado para atender ao requisito de **demonstração mínima do sistema ou pipeline**. Ele executa uma versão didática e autocontida do fluxo analítico usado no projeto, usando os dados já pré-processados incluídos no pacote.

A fonte canônica dos resultados finais continua sendo o código em `src/classifier.py`, `src/evaluate.py` e os CSVs em `reports/`. O objetivo aqui é permitir que o avaliador veja, em poucos blocos, como o pipeline carrega os corpora, aplica os cortes temporais `N = [5, 10, 15, 20, 24, 50]`, treina modelos TF-IDF + LinearSVC e reproduz as etapas E1-E5 em escala demonstrativa.


## Requisitos de execução

Execute este notebook a partir da pasta `demo/`. Se o ambiente não tiver as dependências, instale:

```bash
pip install -r requirements.txt
```

A demo inclui CSVs pequenos e fictícios em `data/processed/`, portanto não requer MongoDB, chamadas LLM/API, derivados do PAN 2012 nem os XMLs originais.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, LeaveOneOut
import warnings
warnings.filterwarnings('ignore')

N_MSGS_LIST = [5, 10, 15, 20, 24, 50]
RANDOM_STATE = 42
BOOTSTRAP_N = 1000
TFIDF_PARAMS = dict(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)


def fbeta_from_pr(precision, recall, beta=0.5):
    b2 = beta ** 2
    denom = b2 * precision + recall
    return 0.0 if denom == 0 else (1 + b2) * precision * recall / denom


def scores(y_true, y_pred):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    return {
        'f05': fbeta_from_pr(p, r, beta=0.5),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': p,
        'recall': r,
    }


def make_model():
    return (
        TfidfVectorizer(**TFIDF_PARAMS),
        LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_STATE),
    )


def get_subset(df, n_msgs):
    mask = df['n_msgs'] == n_msgs
    if 'status' in df.columns:
        mask &= df['status'] != 'guardrail_interrupted'
    return df[mask].copy()


## 1. Carregamento dos dados

Carregamos dados demonstrativos pequenos incluídos em `demo/data/processed/`. O notebook tenta usar parquets quando disponíveis, mas usa CSVs fictícios como fallback para evitar versionar derivados de dados reais.


In [ ]:
from pathlib import Path

def load_demo_table(name):
    candidates = [
        Path('data/processed') / f'{name}.parquet',
        Path('data/processed') / f'{name}.csv',
        Path('../data/processed') / f'{name}.parquet',
        Path('../data/processed') / f'{name}.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            if candidate.suffix == '.parquet':
                return pd.read_parquet(candidate)
            return pd.read_csv(candidate)
    raise FileNotFoundError(f'Arquivo demonstrativo não encontrado: {name}')

pan_df = load_demo_table('pan2012_train')
synth_df = load_demo_table('synthetic')

print('PAN demo:', pan_df.shape, '| labels:', pan_df['label'].value_counts().to_dict())
print('Sintético demo:', synth_df.shape, '| labels:', synth_df['label'].value_counts().to_dict())


## 2. E1 - baseline PAN demonstrativo

O baseline canônico do projeto é calculado por `src/classifier.py` com validação cruzada estratificada, dois classificadores e variantes com/sem undersampling. Para a demonstração mínima, usamos um holdout estratificado 80/20 com TF-IDF + LinearSVC, suficiente para mostrar o mesmo núcleo de classificação aplicado aos cortes temporais.


In [ ]:
rows_e1 = []
print(f"{'N':<4} | {'F0.5':<8} | {'F1':<8} | {'Precisão':<9} | {'Recall':<8}")
print('-' * 48)

for n in N_MSGS_LIST:
    pan_n = get_subset(pan_df, n)
    if pan_n.empty:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        pan_n['text'], pan_n['label'],
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=pan_n['label'],
    )

    vec, clf = make_model()
    clf.fit(vec.fit_transform(X_train), y_train)
    metrics = scores(y_test, clf.predict(vec.transform(X_test)))
    rows_e1.append({'n_msgs': n, **metrics})
    print(f"{n:<4} | {metrics['f05']:<8.4f} | {metrics['f1']:<8.4f} | {metrics['precision']:<9.4f} | {metrics['recall']:<8.4f}")

e1_demo = pd.DataFrame(rows_e1)


## 3. E2 - transferência PAN para sintético

Nesta etapa seguimos a lógica de `src/evaluate.py`: para cada corte `n_msgs`, o modelo é treinado em todo o subset PAN disponível e avaliado no corpus sintético correspondente.


In [ ]:
rows_e2 = []
print(f"{'N':<4} | {'F0.5':<8} | {'F1':<8} | {'Precisão':<9} | {'Recall':<8}")
print('-' * 48)

for n in N_MSGS_LIST:
    pan_n = get_subset(pan_df, n)
    syn_n = get_subset(synth_df, n)
    if pan_n.empty or syn_n.empty:
        continue

    vec, clf = make_model()
    clf.fit(vec.fit_transform(pan_n['text']), pan_n['label'])
    metrics = scores(syn_n['label'], clf.predict(vec.transform(syn_n['text'])))
    rows_e2.append({'n_msgs': n, **metrics})
    print(f"{n:<4} | {metrics['f05']:<8.4f} | {metrics['f1']:<8.4f} | {metrics['precision']:<9.4f} | {metrics['recall']:<8.4f}")

e2_demo = pd.DataFrame(rows_e2)


## 4. E3 - consistência interna do sintético via Leave-One-Out

Reproduzimos a validação Leave-One-Out no corpus sintético e estimamos um intervalo de confiança por bootstrap sobre as predições acumuladas.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
rows_e3 = []
print(f"{'N':<4} | {'F0.5':<8} | {'IC 95%':<18}")
print('-' * 36)

for n in N_MSGS_LIST:
    syn_n = get_subset(synth_df, n)
    X_arr = syn_n['text'].values
    y_arr = syn_n['label'].values
    if len(X_arr) < 2 or len(np.unique(y_arr)) < 2:
        continue

    y_true_loo, y_pred_loo = [], []
    for train_idx, test_idx in LeaveOneOut().split(X_arr):
        if len(np.unique(y_arr[train_idx])) < 2:
            continue
        vec, clf = make_model()
        clf.fit(vec.fit_transform(X_arr[train_idx]), y_arr[train_idx])
        y_pred_loo.extend(clf.predict(vec.transform(X_arr[test_idx])).tolist())
        y_true_loo.extend(y_arr[test_idx].tolist())

    y_true = np.array(y_true_loo)
    y_pred = np.array(y_pred_loo)
    metrics = scores(y_true, y_pred)

    boot = []
    for _ in range(BOOTSTRAP_N):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        p = precision_score(y_true[idx], y_pred[idx], zero_division=0)
        r = recall_score(y_true[idx], y_pred[idx], zero_division=0)
        boot.append(fbeta_from_pr(p, r, beta=0.5))
    ic_lower, ic_upper = np.percentile(boot, [2.5, 97.5])

    rows_e3.append({'n_msgs': n, 'f05': metrics['f05'], 'ic_lower': ic_lower, 'ic_upper': ic_upper})
    print(f"{n:<4} | {metrics['f05']:<8.4f} | [{ic_lower:.4f}, {ic_upper:.4f}]")

e3_demo = pd.DataFrame(rows_e3)


## 5. E4 - diagnóstico lexical por Jaccard

Comparamos as 20 features TF-IDF mais frequentes de cada domínio, por corte temporal, para visualizar a distância lexical entre PAN e corpus sintético.


In [ ]:
rows_e4 = []
TOP_N = 20
print(f"{'N':<4} | {'Jaccard top-20':<16} | {'Comuns':<7}")
print('-' * 36)

for n in N_MSGS_LIST:
    pan_n = get_subset(pan_df, n)
    syn_n = get_subset(synth_df, n)
    if pan_n.empty or syn_n.empty:
        continue

    vec_pan = TfidfVectorizer(ngram_range=(1, 2), max_features=TOP_N).fit(pan_n['text'])
    vec_syn = TfidfVectorizer(ngram_range=(1, 2), max_features=TOP_N).fit(syn_n['text'])
    top_pan = set(vec_pan.get_feature_names_out())
    top_syn = set(vec_syn.get_feature_names_out())
    common = sorted(top_pan & top_syn)
    jaccard = len(common) / len(top_pan | top_syn) if (top_pan | top_syn) else 0.0

    rows_e4.append({'n_msgs': n, 'jaccard': jaccard, 'n_common': len(common), 'common_features': common})
    print(f"{n:<4} | {jaccard:<16.4f} | {len(common):<7}")

e4_demo = pd.DataFrame(rows_e4)


## 6. E5 - aumento de dados

Comparamos o modelo treinado apenas com o treino PAN contra um modelo treinado com `PAN train + sintético`, ambos avaliados no mesmo holdout PAN.


In [ ]:
rows_e5 = []
print(f"{'N':<4} | {'F0.5 baseline':<14} | {'F0.5 aumentado':<15} | {'Delta p.p.':<10}")
print('-' * 60)

for n in N_MSGS_LIST:
    pan_n = get_subset(pan_df, n)
    syn_n = get_subset(synth_df, n)
    if pan_n.empty:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        pan_n['text'], pan_n['label'],
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=pan_n['label'],
    )

    vec_base, clf_base = make_model()
    clf_base.fit(vec_base.fit_transform(X_train), y_train)
    base = scores(y_test, clf_base.predict(vec_base.transform(X_test)))

    if syn_n.empty:
        X_aug, y_aug = X_train, y_train
    else:
        X_aug = pd.concat([X_train, syn_n['text']], ignore_index=True)
        y_aug = pd.concat([y_train, syn_n['label']], ignore_index=True)

    vec_aug, clf_aug = make_model()
    clf_aug.fit(vec_aug.fit_transform(X_aug), y_aug)
    aug = scores(y_test, clf_aug.predict(vec_aug.transform(X_test)))

    delta = (aug['f05'] - base['f05']) * 100
    rows_e5.append({'n_msgs': n, 'f05_baseline': base['f05'], 'f05_augmented': aug['f05'], 'delta_pp': delta})
    print(f"{n:<4} | {base['f05']:<14.4f} | {aug['f05']:<15.4f} | {delta:+.2f}")

e5_demo = pd.DataFrame(rows_e5)


## 7. Resumo da demonstração

As tabelas abaixo ficam disponíveis para inspeção após a execução. Para resultados oficiais e discussão metodológica, consulte os CSVs do projeto e o artigo.


In [ ]:
display({
    'E1_baseline_demonstrativo': e1_demo,
    'E2_transferencia': e2_demo,
    'E3_loo': e3_demo,
    'E4_jaccard': e4_demo,
    'E5_augmentation': e5_demo,
})
